In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Load the cleaned data saved on Day 1 (fast: parquet, not the big CSV)
df = pd.read_parquet("../data/interim/complaints_text.parquet")

In [3]:
cutoff = pd.Timestamp('2025-06-01')
df = df[df["Date received"] < cutoff].copy()
print("Data shape:", df.shape)

Data shape: (855988, 4)


In [6]:
# Sub-product is LEAKAGE: used only for this analysis, never as a model input.
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    raw = pd.read_csv("../data/raw/CFPB.csv",
                      usecols=["Date received", "Product", "Sub-product"],
                      engine="python", on_bad_lines="skip")

In [7]:
old = "Credit reporting, credit repair services, or other personal consumer reports"
print(raw.loc[raw["Product"] == old, "Sub-product"].value_counts())

Sub-product
Credit reporting                  195722
Other personal consumer report       862
Credit repair services               388
Name: count, dtype: int64


In [8]:
old1 = "Debt or credit management"
print(raw.loc[raw["Product"] == old1, "Sub-product"].value_counts())

Sub-product
Debt settlement                                   1193
Credit repair services                            1063
Mortgage modification or foreclosure avoidance      91
Student loan debt relief                            33
Name: count, dtype: int64


In [9]:
RENAME_MAP ={
            "Credit reporting, credit repair services, or other personal consumer reports":
        "Credit reporting or other personal consumer reports", 
     "Payday loan, title loan, or personal loan":
        "Payday loan, title loan, personal loan, or advance loan"   
}

In [10]:
for old in RENAME_MAP:
    print(old in df["Product"].unique(), "|", old)

True | Credit reporting, credit repair services, or other personal consumer reports
True | Payday loan, title loan, or personal loan


In [11]:
RENAME_MAP.values()

dict_values(['Credit reporting or other personal consumer reports', 'Payday loan, title loan, personal loan, or advance loan'])

In [12]:
cards = "Credit card or prepaid card"
print(raw.loc[raw["Product"] == cards, "Sub-product"].value_counts())


Sub-product
General-purpose credit card or charge card    8831
Store credit card                             1091
General-purpose prepaid card                   458
Government benefit card                        256
Gift card                                       88
Payroll card                                    43
Student prepaid card                             2
Name: count, dtype: int64


In [21]:
room_counts = raw.loc[raw["Product"] == cards, "Sub-product"].value_counts()
is_prepaid = room_counts.index.str.contains("prepaid|gift|payroll|benefit", case=False)

prepaid = room_counts[is_prepaid].sum()
credit = room_counts[~is_prepaid].sum()
print(f"Credit: {credit:,} | Prepaid: {prepaid:,} | Prepaid share: {prepaid / (credit + prepaid):.1%}")


Credit: 9,922 | Prepaid: 847 | Prepaid share: 7.9%


In [22]:
# Sub-product is used ONLY to build labels. Never a model input.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lookup = pd.read_csv("../data/raw/CFPB.csv",
                         usecols=["Complaint ID", "Sub-product"],
                         engine="python", on_bad_lines="skip")


In [24]:
df = df.merge(lookup, on ="Complaint ID", how ="left",validate='one_to_one')
print(df.shape)
print("Missing Sub-product:",df['Sub-product'].isna().sum())

(855988, 5)
Missing Sub-product: 0


In [27]:
df["label"] = df["Product"].replace(RENAME_MAP)
print(df["label"].value_counts())


label
Credit reporting or other personal consumer reports        644750
Debt collection                                             63256
Checking or savings account                                 36776
Money transfer, virtual currency, or money service          35114
Credit card                                                 29084
Mortgage                                                    12942
Student loan                                                 9100
Vehicle loan or lease                                        8139
Credit card or prepaid card                                  7194
Payday loan, title loan, personal loan, or advance loan      5156
Prepaid card                                                 3032
Debt or credit management                                    1445
Name: count, dtype: int64


In [28]:
# Card split: sort old "Credit card or prepaid card" rows using Sub-product.
# Sub-product is used ONLY to build labels, never as a model input.
CREDIT_SUBPRODUCTS = ["General-purpose credit card or charge card", "Store credit card"]
PREPAID_SUBPRODUCTS = ["General-purpose prepaid card", "Government benefit card",
                       "Gift card", "Payroll card", "Student prepaid card"]

old_card = df["label"] == "Credit card or prepaid card"

df.loc[old_card & df["Sub-product"].isin(CREDIT_SUBPRODUCTS), "label"] = "Credit card"
df.loc[old_card & df["Sub-product"].isin(PREPAID_SUBPRODUCTS), "label"] = "Prepaid card"

print("Still in old box:", (df["label"] == "Credit card or prepaid card").sum())
print("Unique labels:", df["label"].nunique())


Still in old box: 0
Unique labels: 11


In [29]:
# Decision 2: move credit-repair rows into Debt or credit management.
# Sub-product is used ONLY to build labels, never as a model input.
CREDIT_REPORTING = "Credit reporting or other personal consumer reports"

mask = (df["label"] == CREDIT_REPORTING) & (df["Sub-product"] == "Credit repair services")
print("Rows to move:", mask.sum())

df.loc[mask, "label"] = "Debt or credit management"

print(df["label"].value_counts())
print("Unique labels:", df["label"].nunique())


Rows to move: 193
label
Credit reporting or other personal consumer reports        644557
Debt collection                                             63256
Checking or savings account                                 36776
Credit card                                                 35717
Money transfer, virtual currency, or money service          35114
Mortgage                                                    12942
Student loan                                                 9100
Vehicle loan or lease                                        8139
Payday loan, title loan, personal loan, or advance loan      5156
Prepaid card                                                 3593
Debt or credit management                                    1638
Name: count, dtype: int64
Unique labels: 11


In [30]:
# Texts with more than one label
n_labels = df.groupby("Consumer complaint narrative")["label"].nunique()
conflict_texts = n_labels[n_labels > 1].index
print("Conflicting texts:", len(conflict_texts))

# How many rows those texts cover
rows_affected = df["Consumer complaint narrative"].isin(conflict_texts).sum()
print("Rows affected:", rows_affected)


Conflicting texts: 781
Rows affected: 68431


In [31]:
test_period = df[df["Date received"] >= pd.Timestamp('2024-06-01')].copy()
print(test_period['label'].value_counts())

label
Credit reporting or other personal consumer reports        426643
Debt collection                                             38664
Money transfer, virtual currency, or money service          30399
Checking or savings account                                 21695
Credit card                                                 16541
Mortgage                                                     5943
Student loan                                                 5079
Vehicle loan or lease                                        4070
Payday loan, title loan, personal loan, or advance loan      2699
Prepaid card                                                 1689
Debt or credit management                                    1087
Name: count, dtype: int64


In [32]:
# Sub-product built the labels; it is dropped so it can never become a model feature.
KEEP = ["Complaint ID", "Date received", "Consumer complaint narrative", "Product", "label"]
df[KEEP].to_parquet("../data/interim/complaints_labeled.parquet", index=False)

# Proof: reload what was saved
check = pd.read_parquet("../data/interim/complaints_labeled.parquet")
print(check.shape)
print(check["label"].nunique())


(855988, 5)
11
